# Where does the exact Hessian actually spend its time?

Standalone kernel-level profile. Runs in **~5 minutes** -- it does a 3-iteration solve purely to
capture the real callback, then replays that one call under `torch.profiler`. Nothing here
re-runs the benchmark grid.

### What is already established

`gpu_profile_collocation_runtime.ipynb` localised the cost. On an A100 at `maxiter=100`:

| | gauss-newton | exact |
|---|---|---|
| obj + grad + constraints | 27.8 s | 27.7 s |
| constraint Jacobian | 9.5 s | 9.6 s |
| **Hessian** | 0.5 s | **177.1 s** |
| IPOPT KKT (CPU) | 14.5 s | 19.4 s |
| **total** | **52.3 s** | **233.9 s** |

Exact is Gauss-Newton **plus one call**, and that call is 76% of wall-clock. Everything else is
identical to within noise.

That call achieves only **2.15x** on an A100 -- 3.12x with `TWIN4BUILD_HESS_CHUNK_DIV=1`, which
removes an unconditional halving of the vmap. Because it dominates, nothing else lifts the arm
above roughly that, whatever the Amdahl ceiling says. And the ceiling (21.8x) was answering a
different question: it measures how much work *could* move to the GPU, not how well the GPU runs
it. **Offloadable is not the same as efficiently parallel.**

Three hypotheses have already been tested and settled:

* **fp64 throughput** -- REFUTED. Measured on an A100, where fp64 runs at 1/2 of fp32, not the
  1/32 of a consumer card.
* **unsynchronised CUDA timing** -- REFUTED. `torch_frac` moves by 0.001 with explicit
  `synchronize()`; the device-to-host copy in each callback was already forcing a sync.
* **vmap chunking** -- CONFIRMED, and worth 1.95x on the Hessian call, but only that. It cannot
  close an ~8x gap on its own.

### What is still unknown

Even at `div=1` the exact Hessian uses **12.6%** of its ceiling. Nobody has looked inside the
kernel. This notebook does.

**Setup**: Runtime > Change runtime type > **GPU**, then Run all.

In [ ]:
# --- Setup (Colab-aware) ---------------------------------------------------
# Needs TWIN4BUILD_HESS_CHUNK_DIV, added alongside this notebook.
TWIN4BUILD_REF = "fix/issue-damper-ventilation-identifiability"

try:
    import twin4build as tb
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail loudly now rather than 30 minutes in.
import inspect

import twin4build.estimator._transcription as _tr

_src = inspect.getsource(_tr)
_missing = [n for n in ("TWIN4BUILD_HESS_CHUNK_DIV", "exact_hessian",
                        "boundary_state_init") if n not in _src]
if _missing:
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > Restart session, then re-run this cell)"""
    )

import functools
import os
import time

import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])


def hardware():
    import platform
    import re

    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.lower().startswith("model name"):
                    cpu = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    ram = None
    try:
        with open("/proc/meminfo") as fh:
            ram = int(re.search(r"[0-9]+", fh.readline()).group()) / 1e6
    except OSError:
        pass
    gpu = fp64_ratio = None
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        gpu = f"{pr.name} ({pr.total_memory / 1e9:.0f} GB, sm_{pr.major}{pr.minor})"
        # Datacenter parts (A100/V100, sm_70/80/90) run fp64 at 1/2 of fp32;
        # consumer parts at 1/32-1/64.  Recorded because it was the first
        # hypothesis for the exact-Hessian result and it needs to stay visible.
        fp64_ratio = "1/2 (datacenter)" if pr.major in (7, 8, 9) and pr.minor == 0 \
            else "1/32-1/64 (consumer) -- suspect"
    return {"cpu": cpu, "cores": os.cpu_count(),
            "torch_threads": torch.get_num_threads(),
            "ram_gb": round(ram, 1) if ram else None,
            "gpu": gpu, "fp64": fp64_ratio, "torch": torch.__version__}


HW = hardware()
for k, v in HW.items():
    print(f"{k:14s} {v}")
if not torch.cuda.is_available():
    print("\nNO GPU -- Part A still works; the GPU comparisons will be skipped.")

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples.utils as utils
from twin4build.utils.rgetattr import rgetattr

# `twin4build/examples/full_workflow_example/` (packaged CSVs) is a PACKAGE that
# shadows the module `full_workflow_example.py`, so load the module by path.
import twin4build.examples as _ex_pkg

_p = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_runtime", _p)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20


def build_model(device="cpu", dtype=torch.float64, tag="prof"):
    m = tb.Model(id=f"{tag}_{device}")
    m.load(semantic_model_filename=utils.get_path(
        ["estimator_example", "one_room_example_model.xlsm"]), fcn=fcn)
    m.to(device, dtype)
    return m


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    return [(model.components["office_valve_position_sensor"], 0.05 / 2),
            (model.components["office_temperature_sensor"], 0.1 / 2),
            (model.components["office_damper_position_sensor"], 0.05 / 2),
            (model.components["office_co2_sensor"], 30 / 2)]


COLLOC_OPTS = {"boundary_state_init": "rollout", "early_stopping": False}
print("model builders ready")

## The profile

Parts A-D localised the cost but stopped at the callback boundary. On CUDA at `maxiter=100`
the split is unambiguous:

| | gauss-newton | exact |
|---|---|---|
| obj + grad + constraints | 27.8 s | 27.7 s |
| constraint Jacobian | 9.5 s | 9.6 s |
| **Hessian** | 0.5 s | **177.1 s** |
| IPOPT KKT (CPU) | 14.5 s | 19.4 s |

Exact is Gauss-Newton **plus one call**, and that call is 76% of wall-clock. It achieves 2.15x
on an A100 (3.12x with `HESS_CHUNK_DIV=1`), and because it dominates, nothing else can lift the
arm above roughly that. The 21.8x Amdahl ceiling was answering a different question: it measures
how much work *could* move to the GPU, not how well the GPU runs it. Offloadable is not the same
as efficiently parallel.

This section opens that last box. It captures the real `hess_vals` callback with its real
arguments, replays it under `torch.profiler`, and reports which CUDA kernels consume the time --
plus the launch-bound signature (kernel count and mean duration). The constraint Jacobian is
profiled alongside as a control: it is the same kind of `vmap` over the same segments, one
derivative order lower.

In [ ]:
import twin4build.estimator._casadi_ipopt as _ipopt

if "cuda" in DEVICES:
    from torch.profiler import ProfilerActivity, profile

    # Capture the REAL callbacks with their REAL arguments: replaying the
    # actual closure IPOPT calls avoids profiling a reconstruction of it.
    CAP = {}

    def _grab(fn, key):
        if fn is None:
            return None

        @functools.wraps(fn)
        def w(*a, **kw):
            CAP.setdefault(key, (fn, a, kw))
            return fn(*a, **kw)

        return w

    def capture(hess_div=1, maxiter=3):
        prev = os.environ.get("TWIN4BUILD_HESS_CHUNK_DIV")
        os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = str(hess_div)
        orig = _ipopt.solve_ipopt_constrained

        def wrapped(x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc,
                    options=None, *, hess_vals=None, **kw):
            return orig(x0, lb, ub, fun, grad, n_g, g_fun,
                        _grab(g_jac_vals, "gjac"), jr, jc, options,
                        hess_vals=_grab(hess_vals, "hess"), **kw)

        _ipopt.solve_ipopt_constrained = wrapped
        try:
            model = build_model("cuda", tag="prof")
            est = tb.Estimator(tb.Simulator(model))
            opts = dict(COLLOC_OPTS)
            opts.update({"maxiter": maxiter, "exact_hessian": True})
            est.estimate(START, END, STEP, build_parameters(model),
                         build_measurements(model), n_warmup=N_WARMUP,
                         method=("casadi", "ipopt", "ad", "collocation"), options=opts)
        finally:
            _ipopt.solve_ipopt_constrained = orig
            if prev is None:
                os.environ.pop("TWIN4BUILD_HESS_CHUNK_DIV", None)
            else:
                os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = prev

    capture(hess_div=1)
    print("captured:", sorted(CAP))

    def profile_call(key, repeats=3, row_limit=20):
        if key not in CAP:
            print(f"  {key}: not captured")
            return None
        fn, a, kw = CAP[key]
        fn(*a, **kw)                     # warm up: exclude compile/alloc
        torch.cuda.synchronize()
        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                     record_shapes=False) as prof:
            for _ in range(repeats):
                fn(*a, **kw)
            torch.cuda.synchronize()

        ka = prof.key_averages()
        # torch renamed the CUDA fields to "device"; support both.
        for sort_key in ("self_cuda_time_total", "self_device_time_total"):
            try:
                table = ka.table(sort_by=sort_key, row_limit=row_limit)
                break
            except Exception:
                table = None
        print()
        print(f"===== {key} : top kernels by self CUDA time =====")
        print(table if table else "(could not sort -- printing unsorted)")

        def dev_us(e):
            for attr in ("self_device_time_total", "self_cuda_time_total"):
                v = getattr(e, attr, 0) or 0
                if v:
                    return v
            return 0

        launches = sum(getattr(e, "count", 0) for e in ka if dev_us(e) > 0)
        total_us = sum(dev_us(e) for e in ka)
        print(f"  kernel launches : {launches}")
        print(f"  total device us : {total_us:.0f}  ({total_us / 1e6 / repeats:.3f} s per call)")
        if launches:
            mean = total_us / launches
            print(f"  mean kernel     : {mean:.1f} us")
            print("  => " + ("LAUNCH-BOUND: thousands of microsecond kernels, the batch "
                             "cannot fill the device" if mean < 20 else
                             "COMPUTE-BOUND: kernels are long enough to amortise launch "
                             "overhead; the arithmetic itself is the cost"))
        return ka

    ka_h = profile_call("hess")
    ka_j = profile_call("gjac")   # control: same vmap, one order lower
else:
    print("No GPU -- Part E is CUDA-specific.")

## How to read it

**The discriminator is kernel count and mean duration**, not the top-20 table alone.

* **Thousands of kernels averaging a few microseconds** -> launch-bound. The batch is 360
  segments x 15 states; that is small for an A100, and per-kernel launch latency (~5-10 us)
  dominates the arithmetic. The lever is then a *bigger problem* or *fused kernels* -- not
  scheduling, not fp64, and not chunking. It would also explain why the constraint Jacobian
  (same vmap, one order lower, less work per segment) speeds up even less.
* **A handful of long kernels, dominated by `matrix_exp` or its backward** -> compute-bound. The
  arithmetic itself is the cost, and the levers are fp64 throughput, a cheaper matrix
  exponential, or exploiting structure in the fused state-space block.

**Use the constraint Jacobian as the control.** It is the same `vmap` over the same segments at
one lower derivative order. If both are launch-bound with similar mean kernel durations, the
problem is scale, not the Hessian specifically.

**What would follow from a launch-bound verdict.** The exact Hessian is not "slow" in any
fixable sense at this problem size -- it is doing genuinely more arithmetic than Gauss-Newton
(which is why it converges in fewer iterations), spread over kernels too small to fill the
device. Its case would rest on *iterations saved*, which a fixed-iteration benchmark cannot
reward, and on larger models where the batch finally fills the GPU. That is a scaling
prediction, and `gpu_benchmark_scaling.ipynb` is the notebook that would test it.